# 3 · make — Helico structure-accuracy data

Aggregates the per-target GDT-TS that [Open-Athena/helico](https://github.com/Open-Athena/helico)'s
exp14 published, over the same 333 FoldBench monomers #245 scores. Helico folds each protein from
a contact map; the arms differ only in where the contacts came from, so `Helico, no contacts` and
`Helico + true contacts` bracket what contact conditioning can do at all.

Restricted to targets **every** arm scored, so the arms are compared on one protein set rather
than on whichever ones each of them happened to finish.

CPU only; read anonymously from the public bucket.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "3_gdt_ts"
METRIC = "gdt_ts"      # also available per target: lddt, tm_score, rmsd
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
CLASSES = {"natural": 0, "designed": 1}   # name -> the `designed` flag it selects
PARAMETERS = dict(metric=METRIC, classes=list(CLASSES),
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED)
PARAMETERS

In [ ]:
import io

import pandas as pd

inputs = figlib.Inputs()
frame = pd.read_csv(io.BytesIO(inputs.fetch(f"{figlib.HELICO}/scores/per_target.csv")))
scored = frame[frame.status == "ok"]
complete = scored.groupby("target_id").arm.nunique()
keep = set(complete[complete == scored.arm.nunique()].index)
dropped = sorted(set(scored.target_id) - keep)
per_target = scored[scored.target_id.isin(keep)]
print(f"{scored.arm.nunique()} arms · {len(keep)} targets scored by all of them"
      + (f" · {len(dropped)} dropped: {dropped}" if dropped else ""))

In [ ]:
rows = []
for class_name, designed in CLASSES.items():
    subset = per_target[per_target.designed == designed]
    for arm, group in subset.groupby("arm"):
        mean, low, high = figlib.bootstrap_mean(group[METRIC].values, BOOTSTRAP_DRAWS,
                                                BOOTSTRAP_SEED)
        rows.append(dict(protein_class=class_name, arm=arm, n=len(group),
                         value=mean, ci_low=low, ci_high=high))

summary = pd.DataFrame(rows).sort_values(["protein_class", "value"], ascending=[True, False])
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
figlib.write_dataset(
    DATASET,
    notebook="3_make_gdt_ts_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_target.csv": lambda path: per_target.to_csv(path, index=False),
    },
    extra={
        "metric": {"name": METRIC, "source": "Open-Athena/helico exp14"},
        "arms_dropped_targets": dropped,
        "arms": sorted(per_target.arm.unique()),
    })